In [1]:
import duckdb
import pandas as pd

# Configuration immédiate
duckdb.execute("SET force_download=true")
duckdb.execute("SET threads TO 4")  # Utilise plusieurs threads pour accélérer


In [2]:
# Supprimer les anciennes vues
duckdb.execute("DROP VIEW IF EXISTS titles")
duckdb.execute("DROP VIEW IF EXISTS ratings")
duckdb.execute("DROP VIEW IF EXISTS actors")
duckdb.execute("DROP VIEW IF EXISTS name")
duckdb.execute("DROP VIEW IF EXISTS crew")
duckdb.execute("DROP VIEW IF EXISTS akas")

In [3]:
url_titles = 'https://datasets.imdbws.com/title.basics.tsv.gz'
url_ratings = 'https://datasets.imdbws.com/title.ratings.tsv.gz'
url_actors = 'https://datasets.imdbws.com/title.principals.tsv.gz'
url_akas = "https://datasets.imdbws.com/title.akas.tsv.gz"
url_tmdb = 'D:/Utilisateurs/Dylan Malis/Téléchargements/tmdb_full.csv'
url_crew = "https://datasets.imdbws.com/title.crew.tsv.gz"
url_name = "https://datasets.imdbws.com/name.basics.tsv.gz"

In [4]:
duckdb.query(f"CREATE OR REPLACE VIEW titles AS SELECT * FROM read_csv_auto('{url_titles}')")
duckdb.query(f"CREATE OR REPLACE VIEW tmdb AS SELECT * FROM read_csv_auto('{url_tmdb}')")
duckdb.query(f"CREATE OR REPLACE VIEW akas AS SELECT * FROM read_csv_auto('{url_akas}')")
duckdb.query(f"CREATE OR REPLACE VIEW crew AS SELECT * FROM read_csv_auto('{url_crew}')")
duckdb.query(f"CREATE OR REPLACE VIEW name AS SELECT * FROM read_csv_auto('{url_name}')")
duckdb.query(f"CREATE OR REPLACE VIEW rating AS SELECT * FROM read_csv_auto('{url_ratings}')")



In [5]:
import pandas as pd
pd.set_option('display.max_rows', 500)

query_finale = """
    SELECT titles.tconst, tmdb.original_title, titles.primaryTitle, akas.title, akas.attributes, akas.types, akas.ordering,  titles.startYear,titles.runtimeMinutes,akas.language, tmdb.spoken_languages,titles.genres, name.primaryName,name.primaryProfession, name.knownForTitles, rating.averageRating,tmdb.popularity,tmdb.tagline, tmdb.overview,tmdb.poster_path, tmdb.video, tmdb.budget,tmdb.revenue
    FROM titles
    LEFT JOIN tmdb ON imdb_id = titles.tconst
    LEFT JOIN akas ON akas.titleId = titles.tconst
    LEFT JOIN crew ON crew.tconst = titles.tconst
    LEFT JOIN name ON name.nconst = crew.directors
    LEFT JOIN rating ON rating.tconst = titles.tconst
    WHERE titles.titleType = 'movie'
    AND titles.tconst NOT LIKE '%\\N%'
    AND titles.isAdult = 0
    AND titles.genres IS NOT NULL
    AND titles.genres NOT LIKE '%\\N%'
    AND titles.genres NOT LIKE '%Horror%'
    AND titles.genres NOT LIKE '%NaN%'
    AND titles.runtimeMinutes NOT LIKE '%\\N%'
    AND CAST(titles.runtimeMinutes AS INTEGER) >= 80
    AND CAST(titles.runtimeMinutes AS INTEGER) <= 220
    AND akas.region = 'FR' AND tmdb.spoken_languages LIKE '%fr%'
    AND titles.startYear NOT LIKE '%\\N%'
    AND CAST(titles.startYear as INTEGER) BETWEEN 1960 AND 2027
    AND tmdb.status != 'Canceled'
    AND rating.averageRating >= 5.0

    
"""
# on viens de rajouter le filtre rating, mais pas tester.
#akas.region = 'FR' AND
#AND tmdb.spoken_languages LIKE '%fr%'


In [6]:
# EXÉCUTER LA REQUÊTE ET OBTENIR LE RÉSULTAT
df = duckdb.query(query_finale).to_df()
df.head(50)

,tconst,original_title,primaryTitle,title,attributes,types,ordering,startYear,runtimeMinutes,language,...,primaryProfession,knownForTitles,averageRating,popularity,tagline,overview,poster_path,video,budget,revenue
0,tt0756648,Le Renard et l'Enfant,The Fox and the Child,Le Renard et l'Enfant,\N,imdbDisplay,19,2007,92,\N,...,"director,writer,cinematographer","tt0428803,tt4466550,tt2873250,tt0756648",6.8,7.265,NaN,A young girl of about 10 years lives in a soli...,/eBpc03hS9A7QPv0k6AZfDCdKURt.jpg,False,13000000,29610210
1,tt0758442,Nuit noire,Black Night,Nuit noire,\N,imdbDisplay,5,2005,90,\N,...,"director,writer,producer","tt0758442,tt0236953,tt3613036,tt0219929",6.5,1.495,NaN,"In a world overtaken by eternal darkness, the ...",/o4FEvQXuTKUOFcQNSIim1biac1l.jpg,False,0,0
2,tt0759509,Changement d'adresse,Change of Address,Changement d'adresse,\N,imdbDisplay,9,2006,85,\N,...,"director,writer,actor","tt12443930,tt14831708,tt7530986,tt1700467",6.7,2.767,NaN,"In Paris, the emotional and professional tribu...",/our6U7TsUTXbVWdSNHfupV6Lzlg.jpg,False,0,0
3,tt0762142,Les 3 p'tits cochons,The 3 L'il Pigs,Les 3 p'tits cochons,\N,imdbDisplay,7,2007,124,\N,...,"actor,director,writer","tt0479647,tt1756750,tt4738174,tt0186895",6.9,1.350,NaN,"As their mother lies in a coma, two brothers d...",/tQDU7JCWRZTM8rhPmnNmmQQkBgP.jpg,False,0,4400000
4,tt0765141,La Question humaine,Heartbeat Detector,La question humaine,\N,imdbDisplay,5,2007,143,\N,...,"director,writer,editor","tt0262674,tt0412518,tt7290778,tt0765141",6.2,0.738,NaN,A psychologist discovers troubling links betwe...,/sSvZmKs7ltOOHbUXEWl1CNLb8vP.jpg,False,0,0
5,tt0765142,Qui m'aime me suive,"If You Love Me, Follow Me",Qui m'aime me suive,\N,imdbDisplay,7,2006,102,\N,...,"director,writer,producer","tt2914386,tt1175298,tt0347678,tt0273895",6.1,1.236,NaN,Thirty-five-year-old Max has a successful care...,/9B78LYo6OMXCABunjNX999alEHd.jpg,False,0,0
6,tt0765263,Un ami parfait,Un ami parfait,Un ami parfait,\N,imdbDisplay,2,2006,106,\N,...,"director,actor,writer","tt0086988,tt0284573,tt0072320,tt0117303",5.5,0.840,NaN,"""Julien Rossi wakes up in a hospital bed and f...",/w7z3NPJCZ4sZt0qIr5ssHUmAtSL.jpg,False,0,0
7,tt0765432,Der Baader Meinhof Komplex,The Baader Meinhof Complex,La bande à Baader,\N,imdbDisplay,24,2008,150,\N,...,"director,writer,producer","tt0765432,tt0082176,tt2293522,tt0097714",7.3,10.244,NaN,'Der Baader Meinhof Komplex' depicts the polit...,/4D1MllpbRMt6DDsSCxQxbl2JYM5.jpg,False,27000000,26937355
8,tt0765469,El Pasado,The Past,El pasado,\N,imdbDisplay,4,2007,114,\N,...,"director,writer,producer","tt0293007,tt0082912,tt0120674,tt0089424",6.1,1.005,NaN,"After 12 years, Sofía and Rímini decided to se...",/1BxoNZ6kxoKsr2RUlU17IagvtQE.jpg,False,0,0
9,tt0769508,Dans Paris,Dans Paris,Dans Paris,\N,imdbDisplay,7,2006,89,\N,...,"writer,director,actor","tt0996605,tt7534054,tt1263778,tt32086069",6.3,5.293,NaN,"Paul, depressed from his recent break-up with ...",/92O2VOf6U3qqYTIUmCqaHMqjbW9.jpg,False,0,0


In [7]:
df.describe()

,ordering,averageRating,popularity,budget,revenue
count,9336.000000,9336.000000,9336.000000,9.336000e+03,9.336000e+03
mean,9.610219,6.349625,5.493379,3.226555e+06,8.292680e+06
std,8.509112,0.722326,19.414076,1.618360e+07,5.859733e+07
min,2.000000,5.000000,0.600000,0.000000e+00,0.000000e+00
25%,4.000000,5.800000,1.233000,0.000000e+00,0.000000e+00
50%,7.000000,6.300000,2.528000,0.000000e+00,0.000000e+00
75%,12.000000,6.900000,6.388000,0.000000e+00,0.000000e+00
max,115.000000,9.400000,1170.178000,2.700000e+08,2.264162e+09


In [8]:
df.shape

(9336, 23)

In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 9336 entries, 0 to 9335
Data columns (total 23 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   tconst             9336 non-null   str    
 1   original_title     9336 non-null   str    
 2   primaryTitle       9336 non-null   str    
 3   title              9336 non-null   str    
 4   attributes         9336 non-null   str    
 5   types              9336 non-null   str    
 6   ordering           9336 non-null   int64  
 7   startYear          9336 non-null   str    
 8   runtimeMinutes     9336 non-null   str    
 9   language           9336 non-null   str    
 10  spoken_languages   9336 non-null   str    
 11  genres             9336 non-null   str    
 12  primaryName        8626 non-null   str    
 13  primaryProfession  8626 non-null   str    
 14  knownForTitles     8626 non-null   str    
 15  averageRating      9336 non-null   float64
 16  popularity         9336 non-null   

In [10]:
df.to_csv(
    "resultats_cinema3.csv",
    sep=";",
    index=False,
    encoding="utf-8-sig"
)